# CoNLL-2003 Named Entity Recognition with PyTorch

This notebook is designed for Google Colab and implements an RNN-based NER model using the CoNLL-2003 dataset.

**Contents:**

1. Setup and imports
2. Load and inspect CoNLL-2003 dataset
3. Preprocessing and vectorization
4. PyTorch dataset and dataloader
5. BiLSTM sequence labeling model
6. Training, validation, and evaluation
7. Improvements and next steps


In [ ]:
# Install dependencies when running in Colab
!pip install -q torch torchvision torchaudio datasets seqeval

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Load the CoNLL-2003 dataset
dataset = load_dataset('conll2003')

print(dataset)
print('Train examples:', len(dataset['train']))
print('Validation examples:', len(dataset['validation']))
print('Test examples:', len(dataset['test']))

# Inspect one example
example = dataset['train'][0]
print('Tokens:', example['tokens'])
print('NER tags:', example['ner_tags'])
print('Chunk tags:', example['chunk_tags'])

In [ ]:
# Tag maps and labels
ner_feature = dataset['train'].features['ner_tags']
id2label = ner_feature.int2str
label2id = {label: idx for idx, label in enumerate(ner_feature.feature.names)}

print('Label set:', ner_feature.feature.names)

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
PAD_LABEL = 'O'

def normalize_token(token):
    token = token.lower()
    return token

def parse_sentences(split_dataset):
    sentences = []
    for item in split_dataset:
        tokens = [normalize_token(t) for t in item['tokens']]
        tags = [id2label(tag) for tag in item['ner_tags']]
        sentences.append((tokens, tags))
    return sentences

train_sentences = parse_sentences(dataset['train'])
val_sentences = parse_sentences(dataset['validation'])
test_sentences = parse_sentences(dataset['test'])

print('Sample parsed sentence:', train_sentences[0])

In [ ]:
# Build vocabulary and label maps
from collections import Counter

def build_vocab(sentences, min_freq=1):
    counter = Counter(token for tokens, _ in sentences for token in tokens)
    tokens = [PAD_TOKEN, UNK_TOKEN] + [word for word, freq in counter.items() if freq >= min_freq]
    word2idx = {word: idx for idx, word in enumerate(tokens)}
    return word2idx

word2idx = build_vocab(train_sentences, min_freq=1)
label2idx = {label: idx for idx, label in enumerate(sorted(ner_feature.feature.names))}
idx2label = {idx: label for label, idx in label2idx.items()}

print('Vocabulary size:', len(word2idx))
print('Label map:', label2idx)

def encode_sentence(tokens, tags, word2idx, label2idx):
    input_ids = [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]
    tag_ids = [label2idx[tag] for tag in tags]
    return input_ids, tag_ids

def pad_batch(batch, pad_token_id=0, pad_label_id=None):
    if pad_label_id is None:
        pad_label_id = label2idx[PAD_LABEL]
    max_len = max(len(ids) for ids, tags in batch)
    input_ids = [ids + [pad_token_id] * (max_len - len(ids)) for ids, tags in batch]
    tag_ids = [tags + [pad_label_id] * (max_len - len(tags)) for ids, tags in batch]
    attention_mask = [[1] * len(ids) + [0] * (max_len - len(ids)) for ids, tags in batch]
    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(tag_ids, dtype=torch.long), torch.tensor(attention_mask, dtype=torch.long)

In [ ]:
class NERDataset(Dataset):
    def __init__(self, sentences, word2idx, label2idx):
        self.sentences = sentences
        self.word2idx = word2idx
        self.label2idx = label2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        tokens, tags = self.sentences[idx]
        input_ids, tag_ids = encode_sentence(tokens, tags, self.word2idx, self.label2idx)
        return input_ids, tag_ids

def collate_fn(batch):
    input_ids, tag_ids = zip(*batch)
    return pad_batch(list(zip(input_ids, tag_ids)), pad_token_id=word2idx[PAD_TOKEN], pad_label_id=label2idx[PAD_LABEL])

train_dataset = NERDataset(train_sentences, word2idx, label2idx)
val_dataset = NERDataset(val_sentences, word2idx, label2idx)
test_dataset = NERDataset(test_sentences, word2idx, label2idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print('Example batch sizes:')
for batch in train_loader:
    print([x.shape for x in batch])
    break

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx[PAD_TOKEN])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, attention_mask=None):
        embeddings = self.embedding(input_ids)
        lengths = attention_mask.sum(1).cpu() if attention_mask is not None else None
        packed = torch.nn.utils.rnn.pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False) if lengths is not None else embeddings
        outputs, _ = self.lstm(packed) if lengths is not None else self.lstm(embeddings)
        if lengths is not None:
            outputs, _ = torch.nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        outputs = self.dropout(outputs)
        logits = self.classifier(outputs)
        return logits

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LABELS = len(label2idx)

model = BiLSTMTagger(len(word2idx), EMBEDDING_DIM, HIDDEN_DIM, NUM_LABELS).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=label2idx[PAD_LABEL])
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)

In [ ]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for input_ids, tag_ids, attention_mask in data_loader:
        input_ids = input_ids.to(device)
        tag_ids = tag_ids.to(device)
        attention_mask = attention_mask.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits.view(-1, NUM_LABELS), tag_ids.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for input_ids, tag_ids, attention_mask in data_loader:
            input_ids = input_ids.to(device)
            tag_ids = tag_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids, attention_mask)
            predictions = logits.argmax(-1).cpu().numpy()
            labels = tag_ids.cpu().numpy()
            mask = attention_mask.cpu().numpy()
            batch_size, seq_len = predictions.shape
            for i in range(batch_size):
                preds = [idx2label[predictions[i][j]] for j in range(seq_len) if mask[i][j] == 1]
                refs = [idx2label[labels[i][j]] for j in range(seq_len) if mask[i][j] == 1]
                all_preds.append(preds)
                all_labels.append(refs)
    return all_preds, all_labels

EPOCHS = 3
best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_preds, val_labels = evaluate(model, val_loader, device)
    val_precision = precision_score(val_labels, val_preds)
    val_recall = recall_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds)
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} val_precision={val_precision:.4f} val_recall={val_recall:.4f} val_f1={val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_ner_model.pth')

print('Best validation F1:', best_val_f1)

In [ ]:
# Final evaluation on the test dataset
model.load_state_dict(torch.load('best_ner_model.pth', map_location=device))
test_preds, test_labels = evaluate(model, test_loader, device)
print('Test precision:', precision_score(test_labels, test_preds))
print('Test recall:', recall_score(test_labels, test_preds))
print('Test F1:', f1_score(test_labels, test_preds))
print('
Classification report:
')
print(classification_report(test_labels, test_preds))

## Improvements and next steps

- Experiment with `torch.nn.GRU` instead of `nn.LSTM` for faster training.
- Add character-level embeddings or pretrained word embeddings.
- Use a CRF layer on top of the BiLSTM for structured decoding.
- Increase dropout, use learning rate scheduling, or add weight decay for regularization.
- Save a final model and export inference helper functions for deployment.
